In [1]:
import pandas as pd
import glob
import os

In [ ]:
path_aiming = 'C:/CourseWork/Classifying_grip_strategies_ml/data/01_raw/Aiming/filtered_data'
path_prehension = 'C:/CourseWork/Classifying_grip_strategies_ml/data/01_raw/Prehension/filtered_data'
path_visual_illusion = 'C:/CourseWork/Classifying_grip_strategies_ml/data/01_raw/Visual Illusions/filtered_data'

1. Stage 1: File Discovery and Categorization
    * Purpose: Identify and categorize .csv files into trajectory files (e.g., trajData.csv) and parameter files (everything else like grasp_paramData.csv, etc.).
    * Loops through each dataset directory (e.g., aiming, prehension, etc.).
    * Uses glob to find all .csv files.
    * Categorizes them into:

        * Trajectory files (trajData.csv): contain time-series (kinematic) data.

        * Parameter files: static metadata about trials like subject name, trial number, etc.

    * Reads all parameter files into DataFrames and tags them with a param_source_file column for provenance tracking.

2. Stage 2: Parameter Consolidation
    * Purpose: Merge all static/parameter files into a single table, ensuring one row per unique trial.

    * Concatenates all parameter DataFrames vertically.

    * Groups them by ['subjName', 'trialN'] and keeps only the first occurrence of each pair.

    * Cleans and consolidates static trial metadata across sources.

    * Ensures the param_df is ready to merge with the time-series data.

3. Stage 3: Trajectory Data Merge
    * Purpose: Attach static parameters to every row of time-series data using a left join on ['subjName', 'trialN'].Only the matching rows from the parameter files are merged with each trajectory file, based on two keys subjName and trialN

    * Loops through each trajData.csv file.

    * Reads in trajectory data.

    * Filters relevant rows from param_df for that subject.

    * Drops duplicate columns to avoid merge conflicts.

    * Merges traj_df with the matching static info from param_df.

    * Collects all merged DataFrames into a list.

4. Stage 4: Unify Redundant Sequential Features
    * Purpose: Fix messy columns like indexXraw and indexX by creating a clean _unified column.

    * For each marker (index, thumb, wrist) and axis (X, Y, Z):

        * Checks for both raw and processed versions of the feature.

        * Creates a new _unified column that uses raw if available, otherwise processed.

        * Handles missing values more gracefully by prioritizing available sensor readings.
        
        * Validates how many missing values remain in these unified features.

5. Stage 5: Clean Static Features
    * Purpose: Prepare static features for modeling.

    * One-hot encodes categorical static features like signal.

    * Imputes missing values in numeric static features by filling them with 0.

    * Selects a final list of static features (both numeric + one-hot encoded).

    * Ensures that only existing columns are kept to avoid errors.

6. Stage 6: Engineer Labels (Target Variable)
    * Purpose: Create a grip strategy label based on the experimental setup for supervised ML.

    * Based on dataset_source, it picks relevant columns like visCond, surface, distance, etc.

    * Joins those column values to form a string label: e.g., aiming_clear_black_three.

    * Stores it in the new grip_strategy_label column.

    * Cleans up weird spellings like _woord_ → _wood_.

    * Reports how many unique grip strategy classes exist.

7. Stage 7: Final Selection and Save
    * Purpose: Prepare and save the final clean dataset in Parquet format.

    * Selects only essential columns:

        * Identifiers: subjName, trialN

        * Unified sequential features

        * Selected static features

        * The target label: grip_strategy_label

    * Drops any rows with nulls in sequential features (important for ML models).

    * Saves everything into a compressed Parquet file: comprehensive_master_data_universal.parquet.

In [ ]:
def create_perfect_master_dataset(base_paths):
    """
    Loads, merges, cleans, and unifies all data sources into a single,
    model-ready Parquet file. This version automatically detects and categorizes
    all .csv files.
    """

    # Dynamically Load All Parameter and Trajectory Files 
    # It basically categories all files into 2 groups Traj files and Parameter files. Traj files (trajData.csv) which is considered to contain time series data and other csvs fall under Parameter files (subject name, trial number, experimental conditions, etc)
    print("--- Part 1: Finding and categorizing all .csv files ---")
    
    all_param_files = []
    all_traj_files = []
    
    for dataset_name, base_path in base_paths.items():
        all_csv_files = glob.glob(os.path.join(base_path, '*.csv'))
        print(f"Found {len(all_csv_files)} total .csv files in '{dataset_name}'")
        
        for f in all_csv_files:
            if 'trajData.csv' in os.path.basename(f):
                all_traj_files.append((dataset_name, f))
            else:
                all_param_files.append(f)
                
    print(f"\nIdentified {len(all_param_files)} parameter/static files to merge.")
    print(f"Identified {len(all_traj_files)} trajectory files to process.")

    all_param_dfs = []
    for file_path in all_param_files:
        try:
            df = pd.read_csv(file_path, low_memory=False)
            source_name = os.path.basename(file_path).replace('.csv', '')
            df['param_source_file'] = source_name 
            all_param_dfs.append(df)
        except Exception as e:
            print(f"Could not load or process {file_path}. Error: {e}")
            
    # Combine all parameter and info dataframes together
    param_df = pd.concat(all_param_dfs, ignore_index=True)

    # Consolidate by trial. This is critical for merging data from different
    # files (e.g., grasp_paramData, reach_paramData, AimingData) for the same trial.
    # It assumes 'subjName' and 'trialN' exist in all parameter-like files.
    print("\nConsolidating all parameter data from multiple sources...")
    if 'subjName' in param_df.columns and 'trialN' in param_df.columns:
        param_df = param_df.groupby(['subjName', 'trialN']).first().reset_index()
        print(f"Loaded and consolidated {len(param_df)} unique trial parameter rows.")
    else:
        print("Warning: 'subjName' or 'trialN' not found in all parameter files. Skipping consolidation.")


    # --- Part 2: Process Trajectory Data Iteratively and Merge ---
    # Here, the script merges the time-series (trajectory) data with the corresponding static (parameter) data for each trial.
    # For each trajectory file, it finds the matching parameter data (based on subjName and trialN) and merges it. This adds the static experimental conditions to every single time-step of the trajectory data.

    print("\n--- Part 2: Loading and merging Trajectory (trajData) files one by one ---")
    all_merged_dfs = []
    merge_keys = ['subjName', 'trialN']
    
    print(f"Processing {len(all_traj_files)} trajData files...")
    for dataset_name, file_path in all_traj_files:
        traj_df = pd.read_csv(file_path, low_memory=False)
        traj_df['dataset_source'] = dataset_name
        
        # Merge with the consolidated parameter dataframe
        current_subject = traj_df['subjName'].iloc[0]
        subject_param_df = param_df[param_df['subjName'] == current_subject]
        
        # Avoid duplicate columns after merge
        cols_to_drop_from_params = [col for col in traj_df.columns if col in subject_param_df.columns and col not in merge_keys]
        subject_param_df_unique = subject_param_df.drop(columns=cols_to_drop_from_params, errors='ignore')
        
        merged_df = pd.merge(traj_df, subject_param_df_unique, on=merge_keys, how='left')
        all_merged_dfs.append(merged_df)

    # --- Part 3: Final Concatenation ---
    # All the individually merged DataFrames from Part 2 are combined (stacked vertically) into one large master_df. At this point, all data from all sources is in a single table.
    print("\n--- Part 3: Concatenating all processed pieces ---")
    master_df = pd.concat(all_merged_dfs, ignore_index=True)
    print(f"Total rows in dataframe: {len(master_df)}")

    # --- Part 4: Unifying Sequential Features ---
    # This section focuses on data cleaning for the time-series features. 
    # Problem: The data appears to have both "raw" (e.g., indexXraw) and "processed" (e.g., indexX) columns. Sometimes one might be missing.

    # Solution: It creates a new _unified column (e.g., indexX_unified). It fills this new column with the value from the raw column, and if that value is missing, it uses the value from the proc column. This creates a single, more complete feature.

    print("\n--- Part 4: Unifying sequential features ---")
    markers = ['index', 'thumb', 'wrist']
    axes = ['X', 'Y', 'Z']
    unified_seq_features = []
    for marker in markers:
        for axis in axes:
            raw_col = f'{marker}{axis}raw'
            proc_col = f'{marker}{axis}'
            unified_col = f'{marker}{axis}_unified'
            master_df[unified_col] = master_df[raw_col].fillna(master_df[proc_col])
            unified_seq_features.append(unified_col)
    null_check = master_df[unified_seq_features].isnull().sum()
    print("Nulls remaining in unified sequential features:\n", null_check)

    # --- Part 5: Cleaning and Selecting Static Features
    # This part prepares the non-time-series (static) features for a machine learning model.
    # One-Hot Encoding: It converts categorical columns (like signal) into a numerical format using pd.get_dummies. This is a standard procedure for most machine learning algorithms.
    # Imputation: It fills any missing values in the selected numeric static feature columns with 0.
    # Selection: It defines a final list of static features to keep in the dataset.
    print("\n--- Part 5: Cleaning and selecting static features ---")
    categorical_static = ['signal'] 
    master_df = pd.get_dummies(master_df, columns=categorical_static, prefix=categorical_static, dummy_na=True)
    numeric_static = ['FX', 'FY', 'FZ', 'FVel', 'FAcc', 'MVel', 'MAcc', 'MDec', 'pathLength', 'MGA', 'timeMGA', 'movTime']
    encoded_cols = [col for col in master_df.columns if any(cat in col for cat in categorical_static)]
    final_static_features = numeric_static + encoded_cols
    
    # Ensure all selected static feature columns exist before filling NA
    existing_static_features = [col for col in final_static_features if col in master_df.columns]
    missing_static_features = set(final_static_features) - set(existing_static_features)
    if missing_static_features:
        print(f"Warning: The following expected static columns were not found and will be ignored: {missing_static_features}")
        
    master_df[existing_static_features] = master_df[existing_static_features].fillna(0)
    print(f"Selected {len(existing_static_features)} static features.")

    # --- Part 6: Engineering the Label
    # This feature engineering step creates the target variable (the "label" or "class") for a supervised machine learning model.

    # Label Creation: It defines a set of rules (label_config) to create a unique, descriptive label for each trial. The label is a combination of the dataset source and its key experimental conditions (e.g., aiming_visCond_surface_distance).

    # Cleaning: It performs some minor text cleaning on the generated labels to ensure they are consistent.
    print("\n--- Part 6: Engineering and cleaning labels ---")
    label_config = {
        'aiming': ['visCond', 'surface', 'distance'],
        'prehension': ['visCond', 'surface', 'distance'],
        'visual_illusion': ['visCond', 'illusion', 'targetPos', 'targetSize']
    }
    master_df['grip_strategy_label'] = ''
    for dataset_name, cols in label_config.items():
        if all(c in master_df.columns for c in cols):
            mask = master_df['dataset_source'] == dataset_name
            if mask.sum() > 0:
                # master_df.loc[mask, cols]: Selects the visCond, surface, and distance columns for our row. The values are clear, wood, and 30.

                # .fillna('NA'): If any of these were empty, they would be replaced with the text 'NA'. (Not needed for this example row).

                # .astype(str): Converts all values to text. So, the number 30 becomes the string '30'.

                # .agg('_'.join, axis=1): For each row (axis=1), it joins the values together with an underscore _.

                # condition dataset_source visCond surface distance
                #            aiming         clear   wood    30
                conditions_str = master_df.loc[mask, cols].fillna('NA').astype(str).agg('_'.join, axis=1)
                master_df.loc[mask, 'grip_strategy_label'] = f"{dataset_name}_" + conditions_str
    
    master_df['grip_strategy_label'] = master_df['grip_strategy_label'].str.replace('.csv', '', regex=False)
    master_df['grip_strategy_label'] = master_df['grip_strategy_label'].str.replace('_woord_', '_wood_', regex=False)
    print(f"Final number of unique classes: {master_df['grip_strategy_label'].nunique()}")


    #  --- Part 7: Selecting final columns and saving to Parquet 
    # Column Selection: It selects only the columns that are needed for modeling: identifiers (subjName, trialN), the unified sequential features, the cleaned static features, and the engineered label.

    # Final Cleanup: It drops any rows that still have null values in the critical sequential features to ensure the dataset is completely clean.

    # Saving: The final, polished DataFrame is saved as comprehensive_master_data_universal.parquet. Parquet is a highly efficient, column-oriented file format that is much faster for data analysis and model training than traditional CSV files.

    final_identifiers = ['subjName', 'trialN']
    all_possible_columns = final_identifiers + unified_seq_features + existing_static_features + ['grip_strategy_label']
    final_columns = [col for col in all_possible_columns if col in master_df.columns]

    # Create an explicit copy to avoid the warning
    final_df = master_df[final_columns].copy() 

    # drop rows with nulls from the new DataFrame
    final_df.dropna(subset=unified_seq_features, inplace=True)

    print(f"Final dataframe shape after dropping any remaining nulls: {final_df.shape}")

    parquet_filename = "comprehensive_master_data_universal.parquet"
    final_df.to_parquet(parquet_filename, engine='pyarrow', compression='snappy')
    print(f"\nDone! The perfect '{parquet_filename}' has been created.")
    print("It contains only the necessary columns for modeling.")

In [4]:
if __name__ == '__main__':
    try:
        import pyarrow
    except ImportError:
        print("Error: 'pyarrow' library not found. Please install it: pip install pyarrow")
    else:
        path_aiming = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Aiming/filtered_data'
        path_prehension = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Prehension/filtered_data'
        path_visual_illusion = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Visual Illusions/filtered_data'

        paths_to_process = {
            'aiming': path_aiming,
            'prehension': path_prehension,
            'visual_illusion': path_visual_illusion
        }

        all_paths_exist = True
        for name, path in paths_to_process.items():
            if not os.path.isdir(path):
                print(f"FATAL ERROR: The path for '{name}' does not exist or is not a directory. Path checked: {path}")
                all_paths_exist = False

        if all_paths_exist:
            print("All paths found. Starting dataset creation...")
            create_perfect_master_dataset(paths_to_process)
        else:
            print("\nAborting due to missing paths. Please correct the paths in the script.")


All paths found. Starting dataset creation...
--- Part 1: Finding and categorizing all .csv files ---
Found 73 total .csv files in 'aiming'
Found 81 total .csv files in 'prehension'
Found 80 total .csv files in 'visual_illusion'

Identified 176 parameter/static files to merge.
Identified 58 trajectory files to process.

Consolidating all parameter data from multiple sources...
Loaded and consolidated 2876 unique trial parameter rows.

--- Part 2: Loading and merging Trajectory (trajData) files one by one ---
Processing 58 trajData files...

--- Part 3: Concatenating all processed pieces ---
Total rows in dataframe: 2836573

--- Part 4: Unifying sequential features ---
Nulls remaining in unified sequential features:
 indexX_unified    0
indexY_unified    0
indexZ_unified    0
thumbX_unified    0
thumbY_unified    0
thumbZ_unified    0
wristX_unified    0
wristY_unified    0
wristZ_unified    0
dtype: int64

--- Part 5: Cleaning and selecting static features ---
Selected 14 static featur